# Multimodal drowsiness detection (fusion)

Combines the modalities into **one drowsiness level**. Today: the two camera branches (**EAR** and the eye-state **CNN**). **EOG** and **EEG** have a place reserved and plug in without changing anything else.

```
camera / video file ──► EAR branch ─┐
                   └──► CNN branch ─┤
EOG recording / stream ► EOG branch ┤ (later)   each gives  score 0..1  +  confidence 0..1
EEG recording / stream ► EEG branch ┘
                                     │
   weighted vote:  fused = Σ trust·confidence·score / Σ trust·confidence
                                     ▼
   smoothing (moving average ~3 s) ─► hysteresis (on 0.6 / off 0.4) ─► persistence (3 s above 0.6)
                                     ▼
   ALERT  /  EARLY WARNING  /  DROWSY (sound alarm)  /  MICROSLEEP (immediate alarm)   or   SENSORS UNRELIABLE
```
* **Switch any modality off** in the config cell (`enabled`), or press `1`-`4` in the video window while it runs.
* **Pre-recorded or live:** the same code. A video file is replayed with its own timestamps, a camera uses the clock. EOG/EEG replays will use the same time base.
* The code of the branches is in `drowsiness/video/`, the fusion rules in `drowsiness/fusion/fuse.py` (unit-tested), this notebook shows the settings, the display and the results.

## 1. Settings

In [1]:
import os
import subprocess
import sys
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

from drowsiness.bio.eeg_branch import EegBranch
from drowsiness.bio.eog_branch import EogBranch
from drowsiness.fusion.fuse import ALERT, DROWSY, EARLY_WARNING, MICROSLEEP, UNRELIABLE, Fusion
from drowsiness.video.capture import open_frames
from drowsiness.video.cnn_branch import CnnBranch
from drowsiness.video.ear_branch import EarBranch
from drowsiness.video.hud import draw_face_box, put_text

# Video file, camera index (0 = laptop camera) or stream URL. Later also a recorded EOG/EEG file.
SOURCE = "data/test_vid_1.mov"

# Which modalities take part, and how much each is trusted (the weight w in  w * confidence * score).
# EOG and EEG are placeholders for now: switching them on gives "no data" and they are ignored.
MODALITIES = {
    "ear": {"enabled": True, "trust": 0.25, "options": {"calibrate_s": 10}},  # EAR learns your open-eye EAR for 10 s
    "cnn": {"enabled": True, "trust": 0.25},
    "eog": {"enabled": False, "trust": 0.30},
    "eeg": {"enabled": False, "trust": 0.20},
}

# Anti-flicker settings (seconds / score). Starting values, to be tuned on data.
FUSION = dict(
    smooth_s=3.0,  # exponential average of the fused score over about this many seconds
    drowsy_on=0.6,  # DROWSY starts above this ...
    drowsy_off=0.4,  # ... and only clears below this (hysteresis)
    persist_s=3.0,  # score must stay above drowsy_on this long before DROWSY (persistence)
    microsleep_s=2.0,  # one eye closure this long alerts at once (set 1.5 for a stricter rule)
    alert_hold_s=2.5,  # after the eyes open again the red alarm stays this long, then it stops
    break_s=8.0,  # then the yellow 'take a break' level for this long, then ALERT again
    suppress_s=30.0,  # the branch scores remember 30 s: for this long old data cannot restart the alarm
    min_usable_weight=0.10,  # below this total (trust x confidence) the output is "SENSORS UNRELIABLE", not a guess
)

SOUND = True  # macOS alarm sound for DROWSY and MICROSLEEP

## 2. Branches on or off

`get_branch` builds a branch the first time it is needed. A branch that is off is not run and not shown, and fusion renormalises the weights of the rest.

In [2]:
BRANCH_CLASSES = {"ear": EarBranch, "cnn": CnnBranch, "eog": EogBranch, "eeg": EegBranch}


def get_branch(branches, name, options=None):
    if name not in branches:
        branches[name] = BRANCH_CLASSES[name](**(options or {}))
        branches[name].start()
    return branches[name]

## 3. Display: levels and the big red alert

| situation | level | what you see |
|---|---|---|
| fused score below 0.4 | ALERT | green banner |
| 0.4 to 0.6 | EARLY WARNING | yellow banner: take a break soon |
| above 0.6 for 3 s | DROWSY | **big red message + sound alarm** |
| one long closure (2 s) | MICROSLEEP | **big red flashing message + immediate alarm** |
| sensors too weak | SENSORS UNRELIABLE | grey banner, no guess |

**When the driver wakes up.** The alarm does not run on forever: once the eyes are open again for `alert_hold_s` (2.5 s) the red message stops, the **yellow "take a break" banner** stays for `break_s` (8 s), then the level returns to ALERT.
The branch scores still remember the last 30 s (PERCLOS, longest closure), so for `suppress_s` (30 s) after the recovery the score rules cannot restart the alarm from that old data. A new long closure (1 s or more) wakes them up again at once, and a microsleep is always immediate.

In [3]:
LEVEL_COLOUR = {ALERT: (0, 160, 0), EARLY_WARNING: (0, 200, 255), DROWSY: (0, 0, 255), MICROSLEEP: (0, 0, 255), UNRELIABLE: (110, 110, 110)}
LEVEL_TEXT = {
    ALERT: "ALERT",
    EARLY_WARNING: "TAKE A BREAK SOON",
    DROWSY: "DROWSY - WAKE UP!",
    MICROSLEEP: "MICROSLEEP!!! WAKE UP!",
    UNRELIABLE: "SENSORS UNRELIABLE",
}


def draw_fusion(frame, state, active):
    """Draw the fused result: values on the left, level banner (big and red for DROWSY / MICROSLEEP)."""
    height, width = frame.shape[:2]
    font = float(np.clip(width / 1100, 0.6, 1.5))
    thick = max(2, int(font * 2.5))
    level = state["level"]

    # values: one line per modality, switched-off ones are shown as off
    smoothed = state["smoothed_score"]
    lines = [f"FUSED {smoothed:.2f}  (usable weight {state['usable_weight']:.2f})" if smoothed is not None else "FUSED -"]
    for name in MODALITIES:
        b = state["branches"].get(name)
        if not active.get(name):
            lines.append(f"{name.upper():4s} off")
        elif b["status"] != "ok":
            lines.append(f"{name.upper():4s} {b['status']}")
        else:
            note = "   NOT VISIBLE - ignored" if b["confidence"] < 0.3 else ""
            lines.append(f"{name.upper():4s} score {b['score']:.2f}  conf {b['confidence']:.2f}  vote {100 * b['share']:.0f}%{note}")
    for i, text in enumerate(lines):
        put_text(frame, text, (10, int(28 * font) * (i + 1)), font * 0.7, thick=thick)

    colour = LEVEL_COLOUR[level]
    if level in (DROWSY, MICROSLEEP):  # big red message
        if level == MICROSLEEP and int(state["time"] * 3) % 2:
            colour = (0, 0, 140)  # flashing
        text = LEVEL_TEXT[level]
        size = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 1.0, 3)[0]
        scale = 0.92 * width / size[0]
        band = int(size[1] * scale * 2.0)
        top = height - band  # at the bottom, so the face stays visible
        cv2.rectangle(frame, (0, top), (width, top + band), colour, -1)
        cv2.putText(
            frame,
            text,
            (int(width * 0.04), top + int(band * 0.68)),
            cv2.FONT_HERSHEY_SIMPLEX,
            scale,
            (255, 255, 255),
            max(3, int(scale * 2.5)),
        )
    else:  # small banner at the bottom
        band = int(60 * font)
        top = height - band
        cv2.rectangle(frame, (0, top), (width, height), colour, -1)
        put_text(frame, LEVEL_TEXT[level], (10, height - int(band * 0.3)), font * 1.1, thick=thick)

    put_text(frame, state["reason"][:110], (10, top - 10), font * 0.55, thick=max(1, thick - 1))


_last_alarm = [0.0]


def play_alarm(level, t):
    """Sound for DROWSY and MICROSLEEP, repeated every 2 s (macOS system sound)."""
    if level in (DROWSY, MICROSLEEP) and t - _last_alarm[0] >= 2.0 and sys.platform == "darwin":
        _last_alarm[0] = t
        subprocess.Popen(["afplay", "/System/Library/Sounds/Sosumi.aiff"])

## 4. The loop (works for a recording and for live)

For every frame: each active branch updates, fusion combines their latest outputs, the result is drawn, logged and (optionally) sounded.
`transform(t, frame)` can change the frame on purpose (used for the failure tests below).

In [4]:
def run_fusion(
    source=SOURCE,
    modalities=MODALITIES,
    show="notebook",
    save_to=None,
    log_csv=None,
    sound=False,
    max_seconds=None,
    transform=None,
    display_width=960,
):
    """Returns a DataFrame with one row per frame (levels, scores, confidences, votes); log_csv also saves it."""
    active = {name: m["enabled"] for name, m in modalities.items()}
    branches = {}
    for name in active:
        if active[name]:
            get_branch(branches, name, modalities[name].get("options"))
    fusion = Fusion(trust={n: m["trust"] for n, m in modalities.items()}, **FUSION)
    rows, handle, writer, t0 = [], None, None, time.perf_counter()

    try:
        for t, frame in open_frames(source, max_seconds):
            if transform:
                frame = transform(t, frame)

            outputs = {}
            for name in active:
                if active[name]:
                    get_branch(branches, name, modalities[name].get("options")).update(t, frame)
                    outputs[name] = branches[name].latest()
            state = fusion.step(t, outputs)

            row = {
                "t": t,
                "level": state["level"],
                "fused": state["fused_score"],
                "smoothed": state["smoothed_score"],
                "usable_weight": state["usable_weight"],
                "confidence": state["confidence"],
                "reason": state["reason"],
            }
            for name, b in state["branches"].items():
                row.update(
                    {
                        f"{name}_score": b["score"],
                        f"{name}_conf": b["confidence"],
                        f"{name}_share": b["share"],
                        f"{name}_closed_s": b["raw"].get("current_closure_s", 0.0),
                    }
                )
            rows.append(row)

            if sound:
                play_alarm(state["level"], t)
            if show or save_to:
                shown = [branches[name] for name in active if active[name]]
                for branch in shown:
                    branch.draw(frame)  # each branch's own marks (eye points / squares)
                with_box = [b for b in shown if b.last is not None]
                if with_box:  # one face box, coloured by the most confident branch
                    best = max(with_box, key=lambda b: b.last["confidence"]).last
                    draw_face_box(frame, best["face_box"], best["state"])
                draw_fusion(frame, state, active)
                scale = min(1.0, display_width / frame.shape[1])
                small = cv2.resize(frame, None, fx=scale, fy=scale) if scale < 1 else frame
                if save_to and writer is None:
                    writer = cv2.VideoWriter(save_to, cv2.VideoWriter_fourcc(*"mp4v"), 30, (small.shape[1], small.shape[0]))
                if writer is not None:
                    writer.write(small)
                if show == "window":
                    cv2.imshow("fusion (q to quit, 1-4 switch ear/cnn/eog/eeg)", small)
                    key = cv2.waitKey(1) & 0xFF
                    if key == ord("q"):
                        break
                    if key in (ord("1"), ord("2"), ord("3"), ord("4")):
                        name = list(active)[key - ord("1")]
                        active[name] = not active[name]
                elif show == "notebook":
                    picture = Image(data=cv2.imencode(".jpg", small)[1].tobytes())
                    if handle is None:
                        handle = display(picture, display_id=True)
                    else:
                        handle.update(picture)
    except KeyboardInterrupt:
        print("Stopped.")
    finally:
        for branch in branches.values():
            branch.stop()
        if writer is not None:
            writer.release()
        if show == "window":
            cv2.destroyAllWindows()

    df = pd.DataFrame(rows)
    if log_csv and len(df):
        os.makedirs(os.path.dirname(log_csv) or ".", exist_ok=True)
        df.to_csv(log_csv, index=False)
    if len(df):
        print(f"{len(df)} frames, {df.t.iloc[-1]:.1f} s, {len(df) / (time.perf_counter() - t0):.0f} frames per second")
    return df

## 5. Your video file (pre-recorded)
Both camera branches on. Writes an annotated video you can play back.

In [5]:
df = run_fusion(SOURCE, show=None, save_to="runs/fusion_video.mp4", log_csv="runs/fusion_video_log.csv")
print("annotated video: runs/fusion_video.mp4   per-frame log: runs/fusion_video_log.csv\n")


def summary(df):
    """Time spent in each level and the first time each alert level was reached."""
    dt = df.t.diff().fillna(0)
    out = pd.DataFrame({"seconds": dt.groupby(df.level).sum().round(1), "first at (s)": df.groupby("level").t.min().round(1)})
    return out.reindex([l for l in (ALERT, EARLY_WARNING, DROWSY, MICROSLEEP, UNRELIABLE) if l in out.index])


summary(df)

Stream ended or camera stopped delivering frames.
641 frames, 21.6 s, 37 frames per second
annotated video: runs/fusion_video.mp4   per-frame log: runs/fusion_video_log.csv



,seconds,first at (s)
level,,
ALERT,11.6,0.0
EARLY WARNING,4.8,16.6
DROWSY,2.6,14.1
MICROSLEEP,1.2,12.9
SENSORS UNRELIABLE,1.4,1.8


In [6]:
STRIP_LABEL = {EARLY_WARNING: "BREAK", UNRELIABLE: "UNRELIABLE"}  # shorter names inside the narrow colour strip
LEVEL_PLOT = {ALERT: "#2e9e4f", EARLY_WARNING: "#f2b705", DROWSY: "#e34948", MICROSLEEP: "#8e1b1b", UNRELIABLE: "#9aa0a6"}
BRANCH_PLOT = {"ear": "#2a78d6", "cnn": "#eb6834", "eog": "#1baf7a", "eeg": "#4a3aa7"}  # one fixed colour per branch


def level_segments(df):
    """(start, end, level) for every stretch of the same level."""
    segments, start = [], 0
    for i in range(1, len(df) + 1):
        if i == len(df) or df.level.iloc[i] != df.level.iloc[start]:
            segments.append((df.t.iloc[start], df.t.iloc[min(i, len(df) - 1)], df.level.iloc[start]))
            start = i
    return segments


def plot_fusion(df, title="", damage=None):
    """Five panels on one time axis: level, fused score, eyes closed, branch scores, branch confidence.
    damage=(from_s, to_s) shades the stretch that was damaged on purpose in a failure test."""
    names = [n for n in BRANCH_PLOT if f"{n}_conf" in df and df[f"{n}_conf"].max() > 0]
    fig, axes = plt.subplots(5, 1, figsize=(13, 12), sharex=True, gridspec_kw={"height_ratios": [0.55, 2, 1.5, 1.7, 1.5], "hspace": 0.35})
    for ax in axes:
        ax.grid(axis="y", alpha=0.25)
        ax.spines[["top", "right"]].set_visible(False)
        if damage:
            ax.axvspan(*damage, color="#555555", alpha=0.10, lw=0)

    ax = axes[0]  # 1. the level, as one coloured strip with its name written in
    for start, end, level in level_segments(df):
        ax.axvspan(start, end, color=LEVEL_PLOT[level], lw=0)
        if end - start > 0.06 * (df.t.iloc[-1] - df.t.iloc[0]):
            ax.text(
                (start + end) / 2,
                0.5,
                STRIP_LABEL.get(level, level),
                ha="center",
                va="center",
                color="white",
                fontsize=9,
                fontweight="bold",
                transform=ax.get_xaxis_transform(),
            )
    ax.set_yticks([])
    ax.set_ylabel("level", rotation=0, ha="right", va="center")
    ax.grid(False)
    ax.set_title(title or "Fusion result over time", loc="left", fontsize=13, fontweight="bold")

    ax = axes[1]  # 2. what the level is decided from
    ax.plot(df.t, df.fused, color="#9aa0a6", lw=1, label="fused score (every frame)")
    ax.plot(df.t, df.smoothed, color="#202124", lw=2.5, label="smoothed (decides the level)")
    ax.axhline(0.6, color="#e34948", ls=":", lw=1.2)
    ax.text(df.t.iloc[-1], 0.61, "DROWSY starts above 0.6", ha="right", va="bottom", fontsize=8, color="#e34948")
    ax.axhline(0.4, color="#b58900", ls=":", lw=1.2)
    ax.text(df.t.iloc[-1], 0.41, "warning above 0.4", ha="right", va="bottom", fontsize=8, color="#b58900")
    ax.set_ylim(-0.02, 1.02)
    ax.set_ylabel("fused score\n0 alert, 1 drowsy")
    ax.legend(loc="upper left", fontsize=8, ncol=2, frameon=False)

    ax = axes[2]  # 3. what the eyes did
    for n in names:
        ax.step(df.t, df[f"{n}_closed_s"], where="post", color=BRANCH_PLOT[n], lw=1.8, label=n.upper())
    ax.axhline(1.0, color="#e34948", ls=":", lw=1.2)
    ax.text(df.t.iloc[-1], 1.03, "1 s: DROWSY", ha="right", va="bottom", fontsize=8, color="#e34948")
    ax.axhline(2.0, color="#8e1b1b", ls=":", lw=1.2)
    ax.text(df.t.iloc[-1], 2.03, "2 s: MICROSLEEP", ha="right", va="bottom", fontsize=8, color="#8e1b1b")
    ax.set_ylabel("eyes closed now (s)")
    ax.set_ylim(bottom=0)
    ax.legend(loc="upper left", fontsize=8, ncol=3, frameon=False)

    ax = axes[3]  # 4. each branch's own opinion
    for n in names:
        ax.plot(df.t, df[f"{n}_score"], color=BRANCH_PLOT[n], lw=1.8, label=n.upper())
    ax.set_ylim(-0.02, 1.02)
    ax.set_ylabel("branch score\n0 alert, 1 drowsy")
    ax.legend(loc="upper left", fontsize=8, ncol=3, frameon=False)

    ax = axes[4]  # 5. how much each branch is trusted
    for n in names:
        ax.plot(df.t, df[f"{n}_conf"], color=BRANCH_PLOT[n], lw=1.8, label=n.upper())
    ax.axhline(0.3, color="#555555", ls=":", lw=1)
    ax.text(df.t.iloc[0], 0.31, "below 0.3 a branch is ignored", ha="left", va="bottom", fontsize=8, color="#555555")
    ax.set_ylim(-0.02, 1.02)
    ax.set_ylabel("branch confidence")
    ax.set_xlabel("time (s)")
    ax.legend(loc="upper left", fontsize=8, ncol=3, frameon=False)

    handles = [plt.Rectangle((0, 0), 1, 1, color=LEVEL_PLOT[l]) for l in LEVEL_PLOT]
    fig.legend(
        handles,
        list(LEVEL_PLOT),
        loc="upper right",
        ncol=5,
        fontsize=8,
        frameon=False,
        bbox_to_anchor=(0.99, 0.945),
        title="level colours",
        title_fontsize=8,
    )
    if damage:
        fig.text(0.99, 0.06, "grey band: damaged on purpose", ha="right", fontsize=8, color="#555555")
    plt.show()


plot_fusion(df)

## 6. Switch modalities on and off
The same recording with both cameras, only EAR, and only the CNN. Fusion renormalises the weights, so the output stays on the same 0-1 scale.

In [7]:
def only(*names):
    return {n: {**m, "enabled": n in names} for n, m in MODALITIES.items()}


runs = {"EAR + CNN": df, "EAR only": run_fusion(SOURCE, only("ear"), show=None), "CNN only": run_fusion(SOURCE, only("cnn"), show=None)}
table = {}
for name, d in runs.items():
    s = summary(d)
    table[name] = {
        "first DROWSY (s)": s["first at (s)"].get(DROWSY, np.nan),
        "first MICROSLEEP (s)": s["first at (s)"].get(MICROSLEEP, np.nan),
        "seconds DROWSY": s["seconds"].get(DROWSY, 0.0),
        "seconds UNRELIABLE": s["seconds"].get(UNRELIABLE, 0.0),
        "mean confidence": d.confidence.mean().round(2),
    }
pd.DataFrame(table).T

Stream ended or camera stopped delivering frames.
641 frames, 21.6 s, 96 frames per second


Stream ended or camera stopped delivering frames.
641 frames, 21.6 s, 66 frames per second


,first DROWSY (s),first MICROSLEEP (s),seconds DROWSY,seconds UNRELIABLE,mean confidence
EAR + CNN,14.1,12.9,2.6,1.4,0.54
EAR only,14.1,12.9,2.5,11.4,0.39
CNN only,14.1,12.9,2.5,5.6,0.69


## 7. Failure tests: does it degrade gracefully?
The recording is damaged on purpose between 5 s and 9 s. A good system must not call that stretch "alert" (or "microsleep") by guessing: confidence should drop and the output should say the sensors are unreliable.
Damage: lights out, camera covered, **sunglasses** (dark patch over the eyes, which looks like a closed eye to a network) and a **hand** over the eyes. EAR is set to calibrate for 3 s here so that it is already working at 5 s.

In [8]:
# eye region of the face in this recording, found once with the face detector (for the sunglasses and hand tests)
cap = cv2.VideoCapture(SOURCE)
cap.set(cv2.CAP_PROP_POS_MSEC, 6000)
_, still = cap.read()
cap.release()
detector = cv2.FaceDetectorYN.create("models/face_detection_yunet_2023mar.onnx", "", (still.shape[1], still.shape[0]), 0.6, 0.3, 5000)
face = detector.detect(still)[1][0]
eye_x, eye_y, gap = face[[4, 6]], face[[5, 7]], abs(face[4] - face[6])
x0, x1 = int(eye_x.min() - 0.45 * gap), int(eye_x.max() + 0.45 * gap)
y0, y1 = int(eye_y.mean() - 0.30 * gap), int(eye_y.mean() + 0.30 * gap)


def damage(patch, colour, grow=0.0):
    def apply(t, frame):
        if 5 <= t < 9:
            frame = frame.copy()
            frame[y0 - int(grow * gap) : y1 + int(grow * gap), x0 - int(0.6 * grow * gap) : x1 + int(0.6 * grow * gap)] = colour
        return frame

    return apply


tests = {
    "lights out 5-9 s": lambda t, f: (f * 0.05).astype(np.uint8) if 5 <= t < 9 else f,
    "camera covered 5-9 s": lambda t, f: np.zeros_like(f) if 5 <= t < 9 else f,
    "sunglasses 5-9 s": damage("sunglasses", (25, 25, 25)),
    "hand over eyes 5-9 s": damage("hand", (110, 140, 190), grow=0.5),
}
quick_ear = {**MODALITIES, "ear": {**MODALITIES["ear"], "options": {"calibrate_s": 3}}}

results = {}
for name, transform in tests.items():
    d = run_fusion(SOURCE, quick_ear, show=None, transform=transform)
    results[name] = d
    during = d[(d.t >= 5.5) & (d.t < 9)]
    print(
        f"{name}: levels {during.level.value_counts().to_dict()}, mean confidence EAR {during.ear_conf.mean():.2f} / CNN {during.cnn_conf.mean():.2f}\n"
    )
plot_fusion(results["sunglasses 5-9 s"], title="Sunglasses over the eyes, 5-9 s", damage=(5, 9))
plot_fusion(results["hand over eyes 5-9 s"], title="Hand over the eyes, 5-9 s", damage=(5, 9))

Stream ended or camera stopped delivering frames.
641 frames, 21.6 s, 39 frames per second
lights out 5-9 s: levels {'SENSORS UNRELIABLE': 106}, mean confidence EAR 0.00 / CNN 0.00



Stream ended or camera stopped delivering frames.
641 frames, 21.6 s, 42 frames per second
camera covered 5-9 s: levels {'SENSORS UNRELIABLE': 106}, mean confidence EAR 0.00 / CNN 0.00



Stream ended or camera stopped delivering frames.
641 frames, 21.6 s, 34 frames per second
sunglasses 5-9 s: levels {'SENSORS UNRELIABLE': 106}, mean confidence EAR 0.00 / CNN 0.26



Stream ended or camera stopped delivering frames.
641 frames, 21.6 s, 32 frames per second
hand over eyes 5-9 s: levels {'SENSORS UNRELIABLE': 85, 'ALERT': 21}, mean confidence EAR 0.20 / CNN 0.02



## 8. Live camera
Same function, camera index `0` (laptop camera). `show="window"` opens an OpenCV window where `1`-`4` switch EAR, CNN, EOG, EEG on and off and `q` quits; `show="notebook"` shows frames inline (interrupt the kernel to stop). The EAR branch spends its first 10 s calibrating your eyes, so only the CNN votes until then.
macOS needs camera permission for the app running the notebook.

In [ ]:
live = run_fusion(0, show="notebook", sound=SOUND, save_to="runs/fusion_live.mp4", log_csv="runs/fusion_live_log.csv")

## 9. Adding EOG and EEG later
Nothing here has to change. Build `EogBranch` / `EegBranch` (`drowsiness/bio/`, contract written in their docstrings) so that `latest()` returns `{score, confidence, raw, timestamp}`, set `"enabled": True` in the config cell, and they join the vote with their trust weight. If one of them stops delivering data, fusion marks it `stale` and carries on with the rest.